In [14]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
import json

In [3]:
local_path = "./models/gemma-2b-it"

tokenizer = AutoTokenizer.from_pretrained(local_path)
model = AutoModelForCausalLM.from_pretrained(
    local_path,
    device_map="auto"
)

Loading weights: 100%|██████████| 288/288 [00:00<00:00, 6179.77it/s]


In [ ]:
content = """Industry: Any Industry
Job Title: Administrator (Support & Operations)
Location:
Pune 
Country:
India 
Experience Required: N/A
Primary Skills: N/A
Secondary Skills: N/A
Job Description: Job Summary To independently resolve tickets, provide on call support and doing root cause analysis to ensure positive customer feedback. Key Responsibilities 1. To adhere to quality standards, regulatory requirements and company policies.2. To provide support for on call escalations and doing root cause analysis of given issue.3. Work on value adding activities such Knowledge base update & management, Training freshers, coaching analysts.4. To independently resolve tickets within agreed SLA of ticket volume and time.5. To ensure positive customer experience and CSAT through First Call Resolution and minimum rejected resolutions / Reopen Cases. Skill Requirements Must Have Skills Good to have Skills
Other Requirements 
Additional Requirements: night shifts.
"""

prompt = f"""Convert the above job description this structured JSON:
{content}
"""
tokenizer.pad_token = tokenizer.eos_token

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
input_length = inputs["input_ids"].shape[1]
max_new_tokens = min(1024, int(input_length * 1.5))

outputs = model.generate(
    **inputs,
    max_new_tokens=max_new_tokens,
    do_sample=False,
    eos_token_id=tokenizer.eos_token_id,
    repetition_penalty=1.2,
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Convert the following job description into structured JSON:
Industry: Any Industry
Job Title: Administrator (Support & Operations)
Location:
Pune 
Country:
India 
Experience Required: N/A
Primary Skills: N/A
Secondary Skills: N/A
Job Description: Job Summary To independently resolve tickets, provide on call support and doing root cause analysis to ensure positive customer feedback. Key Responsibilities 1. To adhere to quality standards, regulatory requirements and company policies.2. To provide support for on call escalations and doing root cause analysis of given issue.3. Work on value adding activities such Knowledge base update & management, Training freshers, coaching analysts.4. To independently resolve tickets within agreed SLA of ticket volume and time.5. To ensure positive customer experience and CSAT through First Call Resolution and minimum rejected resolutions / Reopen Cases. Skill Requirements Must Have Skills Good to have Skills
Other Requirements 
Additional Requirements:

In [16]:
content = """Industry: Any Industry
Job Title: Administrator (Support & Operations)
Location:
Pune 
Country:
India 
Experience Required: N/A
Primary Skills: N/A
Secondary Skills: N/A
Job Description: Job Summary To independently resolve tickets, provide on call support and doing root cause analysis to ensure positive customer feedback. Key Responsibilities 1. To adhere to quality standards, regulatory requirements and company policies.2. To provide support for on call escalations and doing root cause analysis of given issue.3. Work on value adding activities such Knowledge base update & management, Training freshers, coaching analysts.4. To independently resolve tickets within agreed SLA of ticket volume and time.5. To ensure positive customer experience and CSAT through First Call Resolution and minimum rejected resolutions / Reopen Cases. Skill Requirements Must Have Skills Good to have Skills
Other Requirements 
Additional Requirements: night shifts.
"""

prompt = f"""{content}
Convert the above job description this structured JSON: 
[
  "job_title": "",
  "location": "",
  "industry": "",
  "responsibilities": [],
  "requirements": [],
  "qualifications": [],
  "experience": [],
  "other_requirements": []
]
"""
tokenizer.pad_token = tokenizer.eos_token

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

input_length = inputs["input_ids"].shape[1]
max_new_tokens = min(768, max(256, int(input_length * 1.2)))

outputs = model.generate(
    **inputs,
    max_new_tokens=max_new_tokens,
    do_sample=False,
    eos_token_id=tokenizer.eos_token_id,
)

output = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(output)

Industry: Any Industry
Job Title: Administrator (Support & Operations)
Location:
Pune 
Country:
India 
Experience Required: N/A
Primary Skills: N/A
Secondary Skills: N/A
Job Description: Job Summary To independently resolve tickets, provide on call support and doing root cause analysis to ensure positive customer feedback. Key Responsibilities 1. To adhere to quality standards, regulatory requirements and company policies.2. To provide support for on call escalations and doing root cause analysis of given issue.3. Work on value adding activities such Knowledge base update & management, Training freshers, coaching analysts.4. To independently resolve tickets within agreed SLA of ticket volume and time.5. To ensure positive customer experience and CSAT through First Call Resolution and minimum rejected resolutions / Reopen Cases. Skill Requirements Must Have Skills Good to have Skills
Other Requirements 
Additional Requirements: night shifts.

Convert the above job description this struc

In [19]:
start = output.find("{")
end = output.rfind("}")

if start != -1 and end != -1:
    output = output[start:end+1]
else:
    output = "{}"

try:
    output_json = json.loads(output)
except:
    output_json = {}

output_json

{'job_title': 'Administrator (Support & Operations)',
 'location': 'Pune, India',
 'industry': 'Any Industry',
 'responsibilities': ['To adhere to quality standards, regulatory requirements and company policies.',
  'To provide support for on call escalations and doing root cause analysis of given issue.',
  'Work on value adding activities such Knowledge base update & management, Training freshers, coaching analysts.',
  'To independently resolve tickets within agreed SLA of ticket volume and time.',
  'To ensure positive customer experience and CSAT through First Call Resolution and minimum rejected resolutions / Reopen Cases.'],
 'requirements': ['Must Have Skills: Good to have Skills',
  'Other Requirements: night shifts.'],
 'qualifications': [],
 'experience': [],
 'other_requirements': ['night shifts.']}

In [24]:
prompts = f"""
You are an expert HR assistant.

Convert the following job description into a structured JSON format.

Rules:
- Extract all relevant information from the input.
- Do not hallucinate missing information.
- If a field is missing, use an empty string "" or empty list [].
- Keep the output strictly in valid JSON format.
- Do not include any explanation or extra text.

Output JSON format:
{{
  "job_title": "",
  "location": "",
  "industry": "",
  "responsibilities": [],
  "requirements": [],
  "qualifications": [],
  "experience": [],
  "other_requirements": []
}}

{content}
"""


inputs = tokenizer(prompts, return_tensors="pt").to(model.device)

input_length = inputs["input_ids"].shape[1]
max_new_tokens = min(768, max(256, int(input_length * 1.2)))

outputs = model.generate(
    **inputs,
    max_new_tokens=max_new_tokens,
    do_sample=False,
    eos_token_id=tokenizer.eos_token_id,
)

output = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(output)


You are an expert HR assistant.

Convert the following job description into a structured JSON format.

Rules:
- Extract all relevant information from the input.
- Do not hallucinate missing information.
- If a field is missing, use an empty string "" or empty list [].
- Keep the output strictly in valid JSON format.
- Do not include any explanation or extra text.

Output JSON format:
{
  "job_title": "",
  "location": "",
  "industry": "",
  "responsibilities": [],
  "requirements": [],
  "qualifications": [],
  "experience": [],
  "other_requirements": []
}

Industry: Any Industry
Job Title: Administrator (Support & Operations)
Location:
Pune 
Country:
India 
Experience Required: N/A
Primary Skills: N/A
Secondary Skills: N/A
Job Description: Job Summary To independently resolve tickets, provide on call support and doing root cause analysis to ensure positive customer feedback. Key Responsibilities 1. To adhere to quality standards, regulatory requirements and company policies.2. To p

In [21]:
start = output.find("{")
end = output.rfind("}")

if start != -1 and end != -1:
    output = output[start:end+1]
else:
    output = "{}"

try:
    output_json = json.loads(output)
except:
    output_json = {}

output_json

{'job_title': '',
 'location': '',
 'industry': '',
 'responsibilities': [],
 'requirements': [],
 'qualifications': [],
 'experience': [],
 'other_requirements': []}

In [25]:
prompt = f"""
You are an expert HR assistant.

Convert the following job description into structured JSON.

Example:
{{
  "job_title": "Enterprise Data Privacy Risk Manager",
  "location": "India",
  "industry": "Information Technology & Services",
  "responsibilities": [
    "Develop and execute comprehensive audit plans for country-level stress testing of data privacy (DP) controls, ensuring thorough assessment and factual accuracy of identified gaps.",
    "Monitor the operating effectiveness of recommended privacy measures and coordinate with internal stakeholders to ensure timely closure of open privacy risks.",
    "Escalate unresolved privacy risks and delays to the Global Privacy Office (GPO) leadership, facilitating prompt resolution and maintaining transparency.",
    "Collaborate with third-party vendor management teams to oversee assessments related to personal data processing, ensuring adherence to privacy policies and standards.",
    "Review, prioritize, and remediate open data privacy risks in partnership with department leads and managers; develop and maintain relevant dashboards for ongoing risk tracking.",
    "Evaluate Data Protection Impact Assessments (DPIAs) for completeness, accuracy, and alignment with regulatory requirements.",
    "Foster a proactive privacy culture through regular communication, training, and awareness initiatives with business units and stakeholders."
  ],
  "requirements": [],
  "qualifications": [],
  "experience": [],
  "other_requirements": [
    "Ability to work collaboratively in a fast-paced, matrixed environment.",
    "Prior experience liaising with third-party vendors and managing cross-functional teams.",
    "Demonstrated commitment to continuous professional development in privacy and risk management domains."
  ]
}}

Input:
Industry: Any Industry
Job Title: N/A
Location: N/A
Country:
India 
Experience Required: N/A
Primary Skills:
Risk Management 
Secondary Skills: N/A
Job Description: Job Summary To ensure implementation of Enterprise Data Privacy program and monitor the operating effectiveness of the measures recommended to manage DP risks. Key Responsibilities 1. To Create The Audit Plan For Country Level Stress Testing Of Dp Related Controls And Review Factual Accuracy Of The Gaps And The Quality Of Testing.2. To Follow-Up With Internal Stakeholders For Closure Of Open Privacy Risks And Escalate To Gpo Leadership For Any Delays Noted From Stakeholders.3. To Liaise With Third Party Vendor Management Spocs And Managers To Oversee Assessment Performed By Associate Pertaining To Personal Data Processing.4. To Review & Prioritize Open Risks For Remediation Along With Department Leads & Manager And Ensure Relevant Dashboards Are Created.5. To Review Data Protection Impact Assessment(Dpias) For Accuracy And Completeness. Skill Requirements Must Have Skills Good to have Skills
Other Requirements 
Additional Requirements: None

Output:
{{
  "job_title": "Python Developer",
  "location": "",
  "industry": "",
  "responsibilities": [],
  "requirements": ["2+ years experience"],
  "qualifications": [],
  "experience": [],
  "other_requirements": []
}}

Now convert:

{content}
"""

tokenizer.pad_token = tokenizer.eos_token

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

input_length = inputs["input_ids"].shape[1]
max_new_tokens = min(768, max(256, int(input_length * 1.2)))

outputs = model.generate(
    **inputs,
    max_new_tokens=max_new_tokens,
    do_sample=False,
    eos_token_id=tokenizer.eos_token_id,
)

output = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(output)


You are an expert HR assistant.

Convert the following job description into structured JSON.

Example:
{
  "job_title": "Enterprise Data Privacy Risk Manager",
  "location": "India",
  "industry": "Information Technology & Services",
  "responsibilities": [
    "Develop and execute comprehensive audit plans for country-level stress testing of data privacy (DP) controls, ensuring thorough assessment and factual accuracy of identified gaps.",
    "Monitor the operating effectiveness of recommended privacy measures and coordinate with internal stakeholders to ensure timely closure of open privacy risks.",
    "Escalate unresolved privacy risks and delays to the Global Privacy Office (GPO) leadership, facilitating prompt resolution and maintaining transparency.",
    "Collaborate with third-party vendor management teams to oversee assessments related to personal data processing, ensuring adherence to privacy policies and standards.",
    "Review, prioritize, and remediate open data privac

In [27]:
start = output.find("{")
end = output.rfind("}")

if start != -1 and end != -1:
    output = output[start:end+1]
else:
    output = "{}"

try:
    output_json = json.loads(output)
except:
    output_json = {}

output_json

{}

In [28]:
SYSTEM_INSTRUCTION = """
You are an AI system designed to transform raw, unstructured job descriptions into structured, professional, and ATS-friendly job descriptions.

Your task:
- Extract relevant information from messy or incomplete job descriptions.
- Organize content into a clean, standardized JSON structure.
- Improve clarity, grammar, and professionalism while preserving meaning.

Rules:
- Output MUST be valid JSON only.
- Follow the exact schema provided.
- Do NOT include explanations or extra text.
- Do NOT hallucinate unrealistic details.
- Infer missing information conservatively.

Writing Guidelines:
- Use clear, concise, professional language.
- Convert vague phrases into actionable statements.
- Remove redundancy and noise.
- Ensure consistency across sections.
- Use bullet-style phrasing for lists.
"""

OUTPUT_SCHEMA = """
{
  "job_title": "",
  "location": "",
  "industry": "",
  "responsibilities": [],
  "requirements": [],
  "qualifications": [],
  "experience": [],
  "other_requirements": []
}
"""

def build_prompt(raw_text):
    return f"""
### TASK:
Convert the following raw job description into a structured and professional JSON format.

### SCHEMA:
{OUTPUT_SCHEMA}

### INPUT:
{raw_text}

### INSTRUCTIONS:
- Clean and structure the content.
- Remove noise and repetition.
- Improve clarity and professionalism.
- Extract meaningful sections.
- Follow the schema strictly.

### OUTPUT:
Return ONLY valid JSON.
"""

In [ ]:
response = model.generate(
    system=SYSTEM_INSTRUCTION,
    prompt=build_prompt(raw_text),
    temperature=0.3,
    top_p=0.9,
    max_tokens=800
)

In [ ]:
def build_prompt_with_examples(raw_text, example_input, example_output):
    return f"""
### TASK:
Convert raw job descriptions into structured JSON.

### SCHEMA:
{OUTPUT_SCHEMA}

### EXAMPLE INPUT:
{example_input}

### EXAMPLE OUTPUT:
{example_output}

### INPUT:
{raw_text}

### OUTPUT:
Return ONLY valid JSON.
"""